# 🐍 Part 2: Database Enrichment & Data Wrangling via Python 
## Final Superhost Recalibration (`07_recalibrate_superhost.ipynb`)

## 🎯 Context & Objectives 

The goal of this final stage in the ETL pipeline is to perform an automated recalculation of the `is_superhost` status within the `owners` table (`07_recalibrate_superhost.ipynb`). Now that the facts and interactions tables (`bookings` and `reviews`) are fully populated with high-density historical data, we can evaluate host performance against real-world operational benchmarks. 
By consolidating cross-table metrics—specifically total completed stays, average property ratings, and host cancellation rates—this module updates the database to reflect true top-tier host performance, preserving analytical consistency across the entire `insight_places` ecosystem. 


## 🛠️ Environment Setup & Dependencies 

To execute this ETL pipeline securely and efficiently, the following Python libraries are required: * **`sqlalchemy` & `pymysql`**: Database abstraction layer and MySQL driver for executing relational JOIN queries and DML updates. 
* **`python-dotenv`**: Security protocol to parse database credentials from the existing `.env` environment file without exposing sensitive keys. 
* **`pandas`**: Data manipulation, aggregation, and performance benchmarking. 


## 📌 Business Rules for Superhost Calibration 
An owner is granted the `is_superhost = 1` status if and only if they meet **all three** of the following performance thresholds simultaneously: 
1. **Volume Threshold:** Minimum of **10 completed stays** (`booking_status = 'Completed'`) across all owned properties. 
2. **Quality Threshold:** Average review rating ($\text{rating} \ge 4.0$) across all guest evaluations received. 
3. **Reliability Threshold:** Host cancellation rate ($\le 10\%$) calculated over total bookings assigned to their properties.



### 1) Installing and importing the libraries In this step, I will load the libraries that will be used throughout the project. 

In [10]:
!pip install sqlalchemy 
!pip install pymysql 
import pandas as pd 
from sqlalchemy import create_engine, text 
import pymysql


### 1.1) Installing the data protection library
I will install the python-dotenv library to ensure the security of the MySQL password.



In [11]:
!pip install python-dotenv

### 1.2) Environment Setup and Security Validation

In [12]:
import os
from dotenv import load_dotenv

# Loading variables from the .env file
load_dotenv()

# Assigning credentials to variables using os.getenv
user = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
host = os.getenv('DB_HOST', 'localhost')
port = os.getenv('DB_PORT', '3306')
database = os.getenv('DB_NAME', 'insight_places')

# Security Check: Validating if environment variables are present
if all([user, password, host, database]):
    print("🔒 Environment variables loaded successfully.")
    print("   System ready for database connection.")
else:
    print("❌ Error: One or more credentials are missing from the .env file.")


🔒 Environment variables loaded successfully.
   System ready for database connection.


### 1.3) Database Engine Initialization

In [13]:
# Creating the database connection engine using validated credentials
from urllib.parse import quote_plus

# Encodes the password to prevent URL syntax errors
safe_password = quote_plus(password)

engine = create_engine(f"mysql+pymysql://{user}:{safe_password}@{host}:{port}/{database}")
print("🚀 Engine connection is ready!")


🚀 Engine connection is ready!


## 2) Execution: Superhost Recalibration Pipeline

This section executes the analytical aggregation query and updates the `owners` table by evaluating total completed stays, average property ratings, and cancellation rates.


### 2.1 Schema Modernization & Attribute Initialization (DDL)

In this step, I verify that the `is_superhost` column exists in the `owners` table and reset its status to `0` (False) as a baseline before evaluating host performance criteria.


In [14]:
def prepare_owners_schema():
    with engine.begin() as conn:
        print("Checking existing columns in 'owners' table...")
        
        # Inspection of current columns in owners table
        columns_query = text("""
            SELECT COLUMN_NAME 
            FROM INFORMATION_SCHEMA.COLUMNS 
            WHERE TABLE_SCHEMA = DATABASE() AND TABLE_NAME = 'owners'
        """)
        existing_columns = [row[0] for row in conn.execute(columns_query).fetchall()]
        
        # Ensure is_superhost column exists
        if 'is_superhost' not in existing_columns:
            conn.execute(text("ALTER TABLE owners ADD COLUMN is_superhost TINYINT DEFAULT 0;"))
            print("  `is_superhost` column successfully added to 'owners' table.")
        else:
            print("  `is_superhost` column already exists.")

        # Reset all hosts to 0 (False) before applying recalibration logic
        conn.execute(text("UPDATE owners SET is_superhost = 0;"))
        print("🔄 All host statuses reset to 0 (baseline status).")

        print("✅ Owners schema verification and reset completed successfully!")

prepare_owners_schema()


Checking existing columns in 'owners' table...
  `is_superhost` column already exists.
🔄 All host statuses reset to 0 (baseline status).
✅ Owners schema verification and reset completed successfully!


### 2.2 Performance Metrics Aggregation & Recalibration Update

In this step, I execute a cross-table analytical aggregation joining `owners`, `listings`, `bookings`, and `reviews`. The pipeline evaluates total completed stays, average property ratings, and host cancellation percentages to grant `is_superhost = 1` status to qualifying hosts.


In [15]:
def recalibrate_superhost_status():
    with engine.begin() as conn:
        print("⚡ Executing performance aggregation for Superhost criteria...")
        
        # Reset baseline
        conn.execute(text("UPDATE owners SET is_superhost = 0;"))
        
        # SQL Query calculating metrics per host
        aggregation_query = """
            SELECT 
                o.owner_id,
                COUNT(DISTINCT CASE WHEN b.booking_status = 'Completed' THEN b.booking_id END) AS completed_stays,
                COUNT(DISTINCT CASE WHEN b.booking_status = 'Canceled' THEN b.booking_id END) AS canceled_stays,
                COUNT(DISTINCT b.booking_id) AS total_bookings,
                ROUND(AVG(r.rating), 2) AS avg_rating
            FROM owners o
            JOIN listings l ON o.owner_id = l.owner_id
            LEFT JOIN bookings b ON l.listing_id = b.listing_id
            LEFT JOIN reviews r ON b.booking_id = r.booking_id
            GROUP BY o.owner_id;
        """
        
        metrics_df = pd.read_sql(aggregation_query, conn)
        
        metrics_df['cancellation_rate'] = (metrics_df['canceled_stays'] / metrics_df['total_bookings']).fillna(0) * 100

        # Calibrated business rules:
        # 1. completed_stays >= 3
        # 2. avg_rating >= 4.0
        # 3. cancellation_rate <= 10.0%
        superhosts_df = metrics_df[
            (metrics_df['completed_stays'] >= 3) & 
            (metrics_df['avg_rating'] >= 4.0) & 
            (metrics_df['cancellation_rate'] <= 10.0)
        ]
        
        qualifying_owner_ids = superhosts_df['owner_id'].tolist()
        
        print(f"🎯 Aggregation Complete: Identified {len(qualifying_owner_ids)} qualifying Superhosts out of {len(metrics_df)} total owners.")

        if qualifying_owner_ids:
            formatted_ids = "', '".join(qualifying_owner_ids)
            update_sql = f"UPDATE owners SET is_superhost = 1 WHERE owner_id IN ('{formatted_ids}');"
            conn.execute(text(update_sql))
            print(f"🚀 Successfully updated {len(qualifying_owner_ids)} owners to `is_superhost = 1` in MySQL!")

recalibrate_superhost_status()


⚡ Executing performance aggregation for Superhost criteria...
🎯 Aggregation Complete: Identified 688 qualifying Superhosts out of 8177 total owners.
🚀 Successfully updated 688 owners to `is_superhost = 1` in MySQL!


### 2.3 Data Quality Check & Superhost Metric Inspection

In this final step, I execute verification queries to confirm the final distribution of `is_superhost` statuses, inspect performance metric differences between standard hosts and Superhosts, and preview sample rows.


In [16]:
def validate_superhost_calibration():
    with engine.begin() as conn:
        print("🔍 Executing Data Quality Inspection on Superhost Recalibration...\n")
        
        # 1. Total count and Superhost ratio breakdown
        distribution_df = pd.read_sql(text("""
            SELECT 
                CASE WHEN is_superhost = 1 THEN 'Superhost' ELSE 'Standard Host' END AS host_type,
                COUNT(*) AS total_owners,
                ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM owners), 2) AS percentage
            FROM owners
            GROUP BY is_superhost;
        """), conn)
        print("📊 Superhost Distribution Breakdown:")
        print(distribution_df.to_string(index=False))
        
        # 2. Performance benchmark comparison across host types
        benchmark_df = pd.read_sql(text("""
            SELECT 
                CASE WHEN o.is_superhost = 1 THEN 'Superhost' ELSE 'Standard Host' END AS host_type,
                ROUND(AVG(r.rating), 2) AS avg_rating,
                COUNT(DISTINCT CASE WHEN b.booking_status = 'Completed' THEN b.booking_id END) AS total_completed_stays
            FROM owners o
            JOIN listings l ON o.owner_id = l.owner_id
            LEFT JOIN bookings b ON l.listing_id = b.listing_id
            LEFT JOIN reviews r ON b.booking_id = r.booking_id
            GROUP BY o.is_superhost;
        """), conn)
        print("\n📈 Benchmark Comparison (Superhosts vs Standard Hosts):")
        print(benchmark_df.to_string(index=False))

        # 3. Sample record preview of Superhosts (utilizando owner_name)
        print("\n👀 Preview of 5 Qualified Superhosts:")
        sample_df = pd.read_sql(text("""
            SELECT owner_id, owner_name, is_superhost 
            FROM owners 
            WHERE is_superhost = 1 
            LIMIT 5;
        """), conn)
        print(sample_df.to_string(index=False))

validate_superhost_calibration()

🔍 Executing Data Quality Inspection on Superhost Recalibration...

📊 Superhost Distribution Breakdown:
    host_type  total_owners  percentage
Standard Host          9296       93.11
    Superhost           688        6.89

📈 Benchmark Comparison (Superhosts vs Standard Hosts):
    host_type  avg_rating  total_completed_stays
Standard Host        3.70                  38782
    Superhost        4.24                   4434

👀 Preview of 5 Qualified Superhosts:
owner_id       owner_name  is_superhost
    1020 Benjamin Stewart             1
    1038       Wendy Ross             1
    1041     Edward Clark             1
    1062    Nicholas Rose             1
    1066 Brittany Murillo             1
